# Unix Shell Tutorial - Protein Publications and Disease Recognition

This is the **sixth tutorial** in a series that will demonstrate how shell scripting can be used to perform the tasks that health and life science specialists may need to undertake to find and retrieve biomedical data and text. We will use the compound caffeine as an example and explore different public repositories to identify diseases related to it. The focus is not on the specific relationships we may discover, but on the process of obtaining them.

The objective of this tutorial is to learn how to automatically retrieve scientific publications associated with proteins and extract text from their titles and abstracts. Additionally, you will learn how to identify and recognize diseases mentioned within the text.

> This tutorial is part of a series of tutorials adapted as interactive versions of the hands-on steps described in the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book, which is licensed under the [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

## Step 01 - Publication URL

Now that we have all the PubMed identifiers, we need to download the text included in the titles and abstracts of each publication.

To retrieve from the UniProt citations service the publication entry of a given identifier, we can again use the `curl` command and a link to the publication entry. For example, if we click on the Format button of the UniProt citations service entry, we can get the link to the RDF/XML version. RDF is a standard data model that can be serialized in a XML format. Thus, in our case, we can deal with this format like we did with XML.

To get started, we first need to retrieve the data file generated in the previous tutorial. The following command downloads the chebi_27732_xrefs_UniProt.csv file directly from the GitHub repository:


In [1]:
%%bash
curl -s -O 'https://raw.githubusercontent.com/lasigeBioTM/data-text-processing-notebooks/refs/heads/main/data/chebi_27732_proteins_xml.zip'
unzip -o chebi_27732_proteins_xml.zip

Archive:  chebi_27732_proteins_xml.zip
  inflating: chebi_27732_A2AGL3.xml  
  inflating: chebi_27732_B0LPN4.xml  
  inflating: chebi_27732_E9PZQ0.xml  
  inflating: chebi_27732_E9Q401.xml  
  inflating: chebi_27732_F1LMY4.xml  
  inflating: chebi_27732_P05177.xml  
  inflating: chebi_27732_P08684.xml  
  inflating: chebi_27732_P21817.xml  
  inflating: chebi_27732_Q13535.xml  
  inflating: chebi_27732_Q15413.xml  
  inflating: chebi_27732_Q8BKX6.xml  
  inflating: chebi_27732_Q8N490.xml  
  inflating: chebi_27732_Q92736.xml  
  inflating: chebi_27732_Q96Q15.xml  
  inflating: chebi_27732_Q9FZ96.xml  
  inflating: chebi_27732_Q9JKK8.xml  


We can retrieve the publication entry by executing the following command:

In [2]:
%%bash
curl https://rest.uniprot.org/citations/1354642.rdf

<?xml version='1.0' encoding='UTF-8'?>
<rdf:RDF xml:base="http://purl.uniprot.org/citations/" xmlns="http://purl.uniprot.org/core/" xmlns:bibo="http://purl.org/ontology/bibo/" xmlns:busco="http://busco.ezlab.org/schema#" xmlns:dcterms="http://purl.org/dc/terms/" xmlns:foaf="http://xmlns.com/foaf/0.1/" xmlns:owl="http://www.w3.org/2002/07/owl#" xmlns:pubmed="http://purl.uniprot.org/pubmed/" xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:rdfs="http://www.w3.org/2000/01/rdf-schema#" xmlns:skos="http://www.w3.org/2004/02/skos/core#" xmlns:up="http://purl.uniprot.org/core/">
<owl:Ontology rdf:about="">
<owl:imports rdf:resource="http://purl.uniprot.org/core/"/>
</owl:Ontology>
<rdf:Description rdf:about="1354642">
<pages>1247-1254</pages>
<volume>13</volume>
<author>Britt B.A.</author>
<author>de Leon S.</author>
<dcterms:identifier>doi:10.1016/0888-7543(92)90042-q</dcterms:identifier>
<author>Duff C.L.</author>
<author>Fujii J.</author>
<name>Genomics</name>
<author>Gillard 

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2735  100  2735    0     0   4249      0 --:--:-- --:--:-- --:--:--  4246


**Expected Output:** RDF/XML data for the publication with ID 1354642

> The `curl` command on this platform has access restrictions but works with `rest.uniprot.org` and `eutils.ncbi.nlm.nih.gov` links.

Alternatively, we can use the web service provided by PubMed at NCBI, by still using curl but with another link:

In [3]:
%%bash
curl 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&id=1354642&retmode=text&rettype=xml'

<?xml version="1.0" ?>
<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January 2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">
<PubmedArticleSet>
<PubmedArticle><MedlineCitation Status="MEDLINE" Owner="NLM" IndexingMethod="Manual"><PMID Version="1">1354642</PMID><DateCompleted><Year>1992</Year><Month>09</Month><Day>22</Day></DateCompleted><DateRevised><Year>2019</Year><Month>09</Month><Day>02</Day></DateRevised><Article PubModel="Print"><Journal><ISSN IssnType="Print">0888-7543</ISSN><JournalIssue CitedMedium="Print"><Volume>13</Volume><Issue>4</Issue><PubDate><Year>1992</Year><Month>Aug</Month></PubDate></JournalIssue><Title>Genomics</Title><ISOAbbreviation>Genomics</ISOAbbreviation></Journal><ArticleTitle>Polymorphisms and deduced amino acid substitutions in the coding sequence of the ryanodine receptor (RYR1) gene in individuals with malignant hyperthermia.</ArticleTitle><Pagination><StartPage>1247</StartPage><EndPage>1254</EndPage><MedlinePg

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  7247    0  7247    0     0  14773      0 --:--:-- --:--:-- --:--:-- 14759


**Expected Output:** XML data containing title and abstract for PubMed ID 1354642

The result is in XML and we can replace the PubMed identifier `1354642` by a comma separated list of identifiers, such as `2298749,1354642,8220422`.

Thus, we can now update the script:

In [4]:
%%bash
cat > getpublications.sh << 'EOF'
# CHEBI identifier given as input is renamed to ID
ID=$1

# Removes any previous files
rm -f chebi_${ID}_*.rdf

grep -l '<name type="scientific">Homo sapiens</name>' chebi_${ID}_*.xml | \
    xargs -I {} \
      grep '<dbReference type="PubMed"' {} | \
    cut -d'"' -f4 | \
    sort -u | \
    xargs -I {} \
      curl -O 'https://rest.uniprot.org/citations/{}.rdf'
EOF

Again, do not forget to save it in our working directory, and add the right permissions with chmod as we did previously with the other scripts.

In [5]:
%%bash
chmod u+x getpublications.sh

**Expected Output:** File permissions updated successfully

In [6]:
%%bash
timeout 10s ./getpublications.sh 27732; echo "Finished or timed out."

Finished or timed out.


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2602  100  2602    0     0   4405      0 --:--:-- --:--:-- --:--:--  4410
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3374  100  3374    0     0   5287      0 --:--:-- --:--:-- --:--:--  5288
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2354  100  2354    0     0   4433      0 --:--:-- --:--:-- --:--:--  4433
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3141  100  3141    0     0   5892      0 --:--:-- --:--:-- --:--:--  5904
  % Total    % Received % Xferd  Average Speed   Tim

**Expected Output:** Downloads multiple RDF files (`chebi_27732_*.rdf`).

> **Note:** We have added a `timeout` because this process takes a significant amount of time due to UniProt API rate limits and access restrictions. Instead of waiting for the script to complete, it is recommended to download the consolidated ZIP file directly from GitHub:

In [7]:
%%bash
curl -s -O 'https://raw.githubusercontent.com/lasigeBioTM/data-text-processing-notebooks/refs/heads/main/data/chebi_27732_publications_rdf.zip'
unzip -o chebi_27732_publications_rdf.zip

Archive:  chebi_27732_publications_rdf.zip
  inflating: 10051009.rdf            
  inflating: 10097181.rdf            
  inflating: 10322772.rdf            
  inflating: 10484775.rdf            
  inflating: 10545197.rdf            
  inflating: 10574461.rdf            
  inflating: 10597277.rdf            
  inflating: 10608806.rdf            
  inflating: 10612851.rdf            
  inflating: 10668853.rdf            
  inflating: 10681376.rdf            
  inflating: 10759686.rdf            
  inflating: 10823104.rdf            
  inflating: 10830164.rdf            
  inflating: 10859164.rdf            
  inflating: 10888602.rdf            
  inflating: 11093772.rdf            
  inflating: 11113224.rdf            
  inflating: 11114888.rdf            
  inflating: 11157710.rdf            
  inflating: 11159812.rdf            
  inflating: 11159936.rdf            
  inflating: 11181494.rdf            
  inflating: 11207026.rdf            
  inflating: 11208676.rdf            
  infla

You can verify the files were created by listing them:

In [8]:
%%bash
ls *.rdf

10051009.rdf
10097181.rdf
10322772.rdf
10484775.rdf
10545197.rdf
10574461.rdf
10597277.rdf
10608806.rdf
10612851.rdf
10668853.rdf
10681376.rdf
10759686.rdf
10823104.rdf
10830164.rdf
10859164.rdf
10888602.rdf
11093772.rdf
11113224.rdf
11114888.rdf
11157710.rdf
11159812.rdf
11159936.rdf
11181494.rdf
11207026.rdf
11208676.rdf
11241852.rdf
11266076.rdf
11295848.rdf
11331269.rdf
11389482.rdf
11418864.rdf
11470508.rdf
11470997.rdf
11525881.rdf
11544179.rdf
11555828.rdf
11575529.rdf
11673449.rdf
11695850.rdf
11709545.rdf
11714865.rdf
11721054.rdf
11726664.rdf
11741831.rdf
11805843.rdf
11865061.rdf
11875366.rdf
11928716.rdf
11948212.rdf
12011431.rdf
12059893.rdf
12066726.rdf
12093772.rdf
12106942.rdf
12112081.rdf
12123492.rdf
12136074.rdf
12168954.rdf
12208234.rdf


**Expected Output:** Lists all downloaded RDF files (chebi_27732_*.rdf)

In the next step, we will extract the titles and abstracts from the RDF files.

## Step 02 - Title and Abstract

Each file has the title and abstract of the publication as values of the `title` and `rdfs:comment` elements, respectively. To extract them we can again use the `xmllint` command.
To install `xmllint` we can execute:

In [9]:
%%bash
apt-get update && apt-get install -y libxml2-utils

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,738 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,611 kB]
Get:14

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


 Now we can execute the `xmllint` command:


In [10]:
%%bash
xmllint --xpath '//*[local-name()="title" or local-name()="comment"]' 10051009.rdf

<title>Mutation screening of the RYR1 gene and identification of two novel mutations in Italian malignant hyperthermia families.</title>
<rdfs:comment>Point mutations in the ryanodine receptor (RYR1) gene are associated with malignant hyperthermia, an autosomal dominant disorder triggered in susceptible people (MHS) by volatile anaesthetics and depolarising skeletal muscle relaxants. To date, 17 missense point mutations have been identified in the human RYR1 gene by screening of the cDNA obtained from muscle biopsies. Here we report single strand conformation polymorphism (SSCP) screening for nine of the most frequent RYR1 mutations using genomic DNA isolated from MHS patients. In addition, the Argl63Cys mutation was analysed by restriction enzyme digestion. We analysed 57 unrelated patients and detected seven of the known RYR1 point mutations. Furthermore, we found a new mutation, Arg2454His, segregating with the MHS phenotype in a large pedigree and a novel amino acid substitution at

**Expected Output:** XML elements containing title and comment text

The output should be the text inside XML elements. To remove the XML elements, we can again add `text()` to the XPath query:

In [11]:
%%bash
xmllint --xpath '//*[local-name()="title" or local-name()="comment"]/text()' 10051009.rdf

Mutation screening of the RYR1 gene and identification of two novel mutations in Italian malignant hyperthermia families.
Point mutations in the ryanodine receptor (RYR1) gene are associated with malignant hyperthermia, an autosomal dominant disorder triggered in susceptible people (MHS) by volatile anaesthetics and depolarising skeletal muscle relaxants. To date, 17 missense point mutations have been identified in the human RYR1 gene by screening of the cDNA obtained from muscle biopsies. Here we report single strand conformation polymorphism (SSCP) screening for nine of the most frequent RYR1 mutations using genomic DNA isolated from MHS patients. In addition, the Argl63Cys mutation was analysed by restriction enzyme digestion. We analysed 57 unrelated patients and detected seven of the known RYR1 point mutations. Furthermore, we found a new mutation, Arg2454His, segregating with the MHS phenotype in a large pedigree and a novel amino acid substitution at position 2436 in another pat

**Expected Output:** Clean text of titles and abstracts without XML tags

Thus, let us create the script `gettext.sh`:

In [12]:
%%bash
cat > gettext.sh << 'EOF'
# CHEBI identifier given as input is renamed to ID
ID=$1

xmllint --xpath '//*[local-name()="title" or local-name()="comment"]/text()' *.rdf
EOF

Again, do not forget to save it in our working directory, and add the right permissions with chmod as we did previously with the other scripts.

In [13]:
%%bash
chmod u+x gettext.sh

**Expected Output:** File permissions updated successfully

In [14]:
%%bash
./gettext.sh 27732 | head -n 10

Mutation screening of the RYR1 gene and identification of two novel mutations in Italian malignant hyperthermia families.
Point mutations in the ryanodine receptor (RYR1) gene are associated with malignant hyperthermia, an autosomal dominant disorder triggered in susceptible people (MHS) by volatile anaesthetics and depolarising skeletal muscle relaxants. To date, 17 missense point mutations have been identified in the human RYR1 gene by screening of the cDNA obtained from muscle biopsies. Here we report single strand conformation polymorphism (SSCP) screening for nine of the most frequent RYR1 mutations using genomic DNA isolated from MHS patients. In addition, the Argl63Cys mutation was analysed by restriction enzyme digestion. We analysed 57 unrelated patients and detected seven of the known RYR1 point mutations. Furthermore, we found a new mutation, Arg2454His, segregating with the MHS phenotype in a large pedigree and a novel amino acid substitution at position 2436 in another pat

**Expected Output:** Displays the first ten lines of the extracted text

We can save the resulting text in a file named `chebi_27732.txt` that we may share or read using our favorite text editor, by adding the redirection operator:

In [15]:
%%bash
./gettext.sh 27732 > chebi_27732.txt

**Expected Output:** Creates chebi_27732.txt file with extracted titles and abstracts

In the next step, we will recognize diseases in the extracted text.

## Step 03 - Disease Recognition

Instead of reading all that text to find any disease related with caffeine, we can try to find sentences about a given disease by using grep:

In [16]:
%%bash
grep 'malignant hyperthermia' chebi_27732.txt

Mutation screening of the RYR1 gene and identification of two novel mutations in Italian malignant hyperthermia families.
Point mutations in the ryanodine receptor (RYR1) gene are associated with malignant hyperthermia, an autosomal dominant disorder triggered in susceptible people (MHS) by volatile anaesthetics and depolarising skeletal muscle relaxants. To date, 17 missense point mutations have been identified in the human RYR1 gene by screening of the cDNA obtained from muscle biopsies. Here we report single strand conformation polymorphism (SSCP) screening for nine of the most frequent RYR1 mutations using genomic DNA isolated from MHS patients. In addition, the Argl63Cys mutation was analysed by restriction enzyme digestion. We analysed 57 unrelated patients and detected seven of the known RYR1 point mutations. Furthermore, we found a new mutation, Arg2454His, segregating with the MHS phenotype in a large pedigree and a novel amino acid substitution at position 2436 in another pat

**Expected Output:** Lines containing 'malignant hyperthermia' from the text file

To save the filtered text in a file named `chebi_27732_hyperthermia.txt`, we only need to add the redirection operator:

In [17]:
%%bash
grep 'malignant hyperthermia' chebi_27732.txt > chebi_27732_hyperthermia.txt

**Expected Output:** Creates chebi_27732_hyperthermia.txt with filtered disease mentions

This is a very simple way of recognizing a disease in text. The next tutorials will describe how to perform more complex text processing tasks.

## Conclusion

This concludes the **Unix Shell** tutorial adapted from the same section of the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book.

In this tutorial, we learned how to retrieve protein-related publications, extract text from titles and abstracts, and identify mentioned diseases.

In the next tutorial in this series will explore more efficient pattern matching techniques to identify diseases in the text.

## Exercise 01

As an exercise, try to identify another disease in the file `chebi_27732.txt`.